In [1]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import plotly.express as px

In [ ]:
example = np.load("../mixed_samples/0healthy.npy")
frame_1 = example[0]

In [26]:
sample = np.load(files[0])
print(files[0].name, sample.shape)

0healthy.npy (10, 18000, 4)


In [3]:
df = pd.DataFrame(frame_1)
df.columns = ['x', 'y', 'z', 'colouring']

In [4]:
pca = PCA()
pca.fit(df)

,"n_components n_components: int, float or 'mle', default=NoneNumber of components to keep.if n_components is not set all components are kept:: n_components == min(n_samples, n_features)If ``n_components == 'mle'`` and ``svd_solver == 'full'``, Minka'sMLE is used to guess the dimension. Use of ``n_components == 'mle'``will interpret ``svd_solver == 'auto'`` as ``svd_solver == 'full'``.If ``0 < n_components < 1`` and ``svd_solver == 'full'``, select thenumber of components such that the amount of variance that needs to beexplained is greater than the percentage specified by n_components.If ``svd_solver == 'arpack'``, the number of components must bestrictly less than the minimum of n_features and n_samples.Hence, the None case results in:: n_components == min(n_samples, n_features) - 1",None
,"copy copy: bool, default=TrueIf False, data passed to fit are overwritten and runningfit(X).transform(X) will not yield the expected results,use fit_transform(X) instead.",True
,"whiten whiten: bool, default=FalseWhen True (False by default) the `components_` vectors are multipliedby the square root of n_samples and then divided by the singular valuesto ensure uncorrelated outputs with unit component-wise variances.Whitening will remove some information from the transformed signal(the relative variance scales of the components) but can sometimeimprove the predictive accuracy of the downstream estimators bymaking their data respect some hard-wired assumptions.",False
,"svd_solver svd_solver: {'auto', 'full', 'covariance_eigh', 'arpack', 'randomized'}, default='auto'""auto"" : The solver is selected by a default 'auto' policy is based on `X.shape` and `n_components`: if the input data has fewer than 1000 features and more than 10 times as many samples, then the ""covariance_eigh"" solver is used. Otherwise, if the input data is larger than 500x500 and the number of components to extract is lower than 80% of the smallest dimension of the data, then the more efficient ""randomized"" method is selected. Otherwise the exact ""full"" SVD is computed and optionally truncated afterwards.""full"" : Run exact full SVD calling the standard LAPACK solver via `scipy.linalg.svd` and select the components by postprocessing""covariance_eigh"" : Precompute the covariance matrix (on centered data), run a classical eigenvalue decomposition on the covariance matrix typically using LAPACK and select the components by postprocessing. This solver is very efficient for n_samples >> n_features and small n_features. It is, however, not tractable otherwise for large n_features (large memory footprint required to materialize the covariance matrix). Also note that compared to the ""full"" solver, this solver effectively doubles the condition number and is therefore less numerical stable (e.g. on input data with a large range of singular values).""arpack"" : Run SVD truncated to `n_components` calling ARPACK solver via `scipy.sparse.linalg.svds`. It requires strictly `0 < n_components < min(X.shape)`""randomized"" : Run randomized SVD by the method of Halko et al... versionadded:: 0.18.0.. versionchanged:: 1.5 Added the 'covariance_eigh' solver.",'auto'
,"tol tol: float, default=0.0Tolerance for singular values computed by svd_solver == 'arpack'.Must be of range [0.0, infinity)... versionadded:: 0.18.0",0.0
,"iterated_power iterated_power: int or 'auto', default='auto'Number of iterations for the power method computed bysvd_solver == 'randomized'.Must be of range [0, infinity)... versionadded:: 0.18.0",'auto'
,"n_oversamples n_oversamples: int, default=10This parameter is only relevant when `svd_solver=""randomized""`.It corresponds to the additional number of random vectors to sample therange of `X` so as to ensure proper conditioning. See:func:`~sklearn.utils.extmath.randomized_svd` for more details... versionadded:: 1.1",10
,"power_iteration_normalizer power_iteration_normalizer: {'auto', 'QR', 'LU', 'none'}, default='auto'Power iteration normalizer for randomized S

In [5]:
sd = np.sqrt(pca.explained_variance_)
prop = np.sqrt(pca.explained_variance_ratio_)
cum = np.cumsum(prop)

summary = pd.DataFrame({
    "Std. dev.": np.sqrt(pca.explained_variance_),
    "Prop. var": pca.explained_variance_ratio_,
    "Cum. prop": np.cumsum(pca.explained_variance_ratio_)
}, index = [f"Component {i + 1}" for i in range(len(pca.explained_variance_ratio_))]).round(6)



In [6]:
loadings = pd.DataFrame(pca.components_.T,
                        index = df.columns,
                        columns = summary.index).round(3)
print("Importance of components:\n", summary, "\n\nloadings:\n", loadings)

Importance of components:
              Std. dev.  Prop. var  Cum. prop
Component 1  34.179512   0.475065   0.475065
Component 2  28.128110   0.321738   0.796804
Component 3  22.344353   0.203029   0.999832
Component 4   0.642098   0.000168   1.000000 

loadings:
            Component 1  Component 2  Component 3  Component 4
x               -0.226        0.948        0.222        0.018
y               -0.068       -0.242        0.968       -0.001
z                0.972        0.203        0.119       -0.000
colouring        0.004       -0.017       -0.003        1.000


In [7]:
folder = Path("../mixed_samples")
files = sorted(folder.glob("*.npy"))

print(f"Found {len(files)} files from mixed_samples")


Found 900 files from mixed_samples


In [8]:
columns = ["x", "y", "z", "colouring"]
frames = []

for path in files:
    data =  np.load(path)

    frame = data[0]

    name = path.stem
    if "healthy" in name:
        label = "healthy"
    elif "pMI" in name:
        label = "pMI"

    frame_df = pd.DataFrame(frame, columns = columns)
    frame_df["sample"] = path.stem
    frame_df["label"] = label
    frames.append(frame_df)

points = pd.concat(frames)
points.head()

,x,y,z,colouring,sample,label
0,-20.157444,17.622805,8.630556,2.0,0healthy,healthy
1,-20.091030,17.621471,8.631974,2.0,0healthy,healthy
2,-20.104204,17.624441,8.617933,2.0,0healthy,healthy
3,-20.120026,17.610435,8.620044,2.0,0healthy,healthy
4,-20.067705,17.655251,8.615248,2.0,0healthy,healthy


In [16]:
feature_names = ["x", "y", "z"]
X = points[feature_names].to_numpy(dtype=float)

pca = PCA()
pca.fit(X)

summary = pd.DataFrame({
    "Std. dev.": np.sqrt(pca.explained_variance_),
    "Prop. var": pca.explained_variance_ratio_,
    "Cum. prop": np.cumsum(pca.explained_variance_ratio_)
}, index = [f"Component {i + 1}" for i in range(len(pca.explained_variance_ratio_))]).round(6)


loadings = pd.DataFrame(pca.components_.T).round(3)
print("Importance of components:\n", summary, "\n\nloadings:\n", loadings)

scores = pca.fit_transform(X)

points["PC1"] = scores[:, 0]
points["PC2"] = scores[:, 1]
points["PC3"] = scores[:, 2]


Importance of components:
              Std. dev.  Prop. var  Cum. prop
Component 1  30.105210   0.468993   0.468993
Component 2  24.964131   0.322490   0.791483
Component 3  20.073779   0.208517   1.000000 

loadings:
        0      1      2
0 -0.225  0.959  0.172
1 -0.129 -0.204  0.970
2  0.966  0.197  0.170


In [21]:
plot_points = points.sample(n=min(10_000, len(points)), random_state=42)

fig = px.scatter_matrix(plot_points, dimensions=["x", "y", "z"], color = "label")
fig.update_traces(diagonal_visible = False, marker = {"size": 3, "opacity": 0.5})
fig.show()

In [ ]:
fig = px.scatter_matrix(plot_points, dimensions=["x", "y", "z"], color = "colouring")
fig.update_traces(diagonal_visible = False, marker = {"size": 3, "opacity": 0.5})
fig.show()

In [22]:
fig = px.scatter_matrix(plot_points, dimensions=["PC1", "PC2", "PC3"], color = "label")
fig.update_traces(diagonal_visible = False, marker = {"size": 3, "opacity": 0.5})
fig.show()

In [24]:
fig = px.scatter_matrix(plot_points, dimensions=["PC1", "PC2", "PC3"], color = "colouring")
fig.update_traces(diagonal_visible = False, marker = {"size": 3, "opacity": 0.5})
fig.show()